In [1]:
from pairs_trading.data.loaders import load_prices
from pairs_trading.stats.cointegration import EGResult, JohansenResult
from pairs_trading.config import SplitConfig

Prepare dictionary of tickers to test for cointegration

In [2]:
ticker_dict = {
    'metals':["GLD", "IAU", "SLV", "GDX"],
    "US":["SPY", "IVV", "VTI"],
    "fixed":["TLT", "IEF", "AGG"],
    "energy":["XLE", "OIH"],
    "finance": ["XLF", "KBE"], 
    "real_estate": ["VNQ", "REM"]
    }

Create the full df with all tickers

In [3]:
all_tickers = [ticker for tickers in ticker_dict.values() for ticker in tickers]
df = load_prices(all_tickers, SplitConfig.train_start, SplitConfig.train_end, refresh=True)

[*********************100%***********************]  16 of 16 completed


### Egle-Granger Tests

In [4]:
from itertools import combinations

for key in ticker_dict.keys():
    pairs = list(combinations(ticker_dict[key], 2))
    print(f"Results for {key}:")
    for pair in pairs:
        result = EGResult(df, pair)
        result.print_results()
    print("-"*40)

Results for metals:
Engle-Granger Result: y = GLD, x=IAU (statistic=-0.1889522939194138, p-value=0.9804017790496286, cointegrated: False)
Engle-Granger Result: y = GLD, x=SLV (statistic=-2.754780416352109, p-value=0.1804972098506913, cointegrated: False)
Engle-Granger Result: y = GLD, x=GDX (statistic=-3.1912986645795542, p-value=0.0713950456640989, cointegrated: False)
Engle-Granger Result: y = IAU, x=SLV (statistic=-3.0439092686262477, p-value=0.10013347507725129, cointegrated: False)
Engle-Granger Result: y = IAU, x=GDX (statistic=-3.3716305710520955, p-value=0.04560059808963391, cointegrated: True)
Engle-Granger Result: y = GDX, x=SLV (statistic=-1.8805243156261047, p-value=0.5898510765082253, cointegrated: False)
----------------------------------------
Results for US:
Engle-Granger Result: y = SPY, x=IVV (statistic=-2.7224458209692592, p-value=0.19160986088904886, cointegrated: False)
Engle-Granger Result: y = VTI, x=SPY (statistic=-0.7351654272808339, p-value=0.9427076172705687,

### Johansen Test

In [10]:
from itertools import combinations

for key in ticker_dict.keys():
    pairs = list(combinations(ticker_dict[key], 2))
    print(f"Results for {key}:")
    for pair in pairs:
        result = JohansenResult(df, pair)
        print(f"{pair[0]}, {pair[1]}:")
        result.print_results()
    print("-"*40)

Results for metals:
GLD, IAU:
Johansen Result: (statistic: 32.09276140604806, critical values:15.4943, cointegrated: True)
GLD, SLV:
Johansen Result: (statistic: 17.399402253793475, critical values:15.4943, cointegrated: True)
GLD, GDX:
Johansen Result: (statistic: 18.69278872565551, critical values:15.4943, cointegrated: True)
IAU, SLV:
Johansen Result: (statistic: 17.480148898828965, critical values:15.4943, cointegrated: True)
IAU, GDX:
Johansen Result: (statistic: 18.107623441577587, critical values:15.4943, cointegrated: True)
SLV, GDX:
Johansen Result: (statistic: 13.174988223610086, critical values:15.4943, cointegrated: False)
----------------------------------------
Results for US:
SPY, IVV:
Johansen Result: (statistic: 122.11142607551746, critical values:15.4943, cointegrated: True)
SPY, VTI:
Johansen Result: (statistic: 7.522867091844633, critical values:15.4943, cointegrated: False)
IVV, VTI:
Johansen Result: (statistic: 7.608783631930745, critical values:15.4943, cointegra

Based on these results I will select pairs that pass one of the tests and are not redundant or trivial (as the many metal combinations). Hence, I pick:

- IAU/GDX
- GLD/SLV
- OIH/XLE
- REM/VNQ (Did not pass the test but had a low p-value of 0.066)
